# Notebook 02: Concurrencia, Asincronía y asyncio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/02_concurrencia_asyncio.ipynb)

**Módulo 16 — Clase 2**

Este notebook acompaña los archivos `03_concurrencia_y_asincronia.md` y `04a_asyncio_fundamentos.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [ ]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

# Jupyter ya tiene un event loop corriendo — podemos usar await directamente en las celdas
# Si usas un script .py, necesitas asyncio.run(main())

## Sección 1: await secuencial vs asyncio.gather — la diferencia central

La diferencia entre M2 (await secuencial) y M4 (gather) es una línea de código.
Medir los tiempos hace la diferencia completamente visible.

In [ ]:
# Tarea simulada con I/O-bound: espera τ segundos
async def tarea_io(nombre: str, duracion: float) -> str:
    # exec(τᵢ): inicializar
    inicio = time.perf_counter()
    # wait(τᵢ): simula I/O (llamada a API, lectura de BD, etc.)
    await asyncio.sleep(duracion)
    # exec(τᵢ): procesar resultado
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0  # cada tarea tarda 1s de I/O
N_TAREAS = 5

# --- M2: await secuencial (esperas NO explotadas) ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

In [ ]:
# --- M4: asyncio.gather (esperas SÍ explotadas) ---
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print()
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

## Sección 2: Traza del event loop con asyncio debug mode

asyncio tiene un modo de depuración que muestra advertencias cuando el event loop se bloquea.
Aquí vemos la diferencia entre `asyncio.sleep` y `time.sleep`.

In [ ]:
import asyncio
import time

# Habilitamos debug mode para ver bloqueos
loop = asyncio.get_event_loop()
loop.set_debug(True)

# Un umbral bajo para detectar bloqueos rápidamente
# (normalmente el umbral es 100ms)
loop.slow_callback_duration = 0.05  # 50ms

# Tarea bien escrita: libera el event loop
async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)   # wait(τ) — event loop libre
    print(f'  {nombre}: fin')

# Tarea mal escrita: BLOQUEA el event loop
async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)            # ← bloquea el hilo del OS entero
    print(f'  {nombre}: fin')

# ¿Qué diferencia ves en la salida?
print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')

In [ ]:
# Desactivar debug mode para el resto del notebook
loop.set_debug(False)

---

## Sección 3: Implementar M2 y M3 — por qué NO mejoran

Implementa los modelos M2 y M3 explícitamente y mide por qué no producen mejora sobre M1 para sus respectivos casos.

In [ ]:
# TAREA 3.1 — M2: async con await secuencial (ya visto en Sección 1)
# Pregunta: ¿por qué M2 es idéntico a M1 en términos de tiempo?
# Respuesta: Porque el operador 'await' detiene la ejecución de la corrutina 
# actual hasta que la tarea finaliza. Al estar dentro de un bucle 'for', 
# el Event Loop no puede iniciar la tarea 'i+1' hasta que la 'i' haya retornado, 
# eliminando cualquier posibilidad de concurrencia.

# Responde con la definición formal: ¿qué condición de M4 falta en M2? 
# Falta el "Concurrent Scheduling" (Programación Concurrente). En M4, tareas como 
# asyncio.gather permiten que exec(τⱼ) ∩ wait(τᵢ) ≠ ∅, es decir, que el CPU procese una tarea 
# mientras otra está en espera. En M2, las esperas NO son explotadas.

# TAREA 3.2 — M3: threading CPU-bound
# Implementa N tareas CPU-bound con threading y mide vs secuencial.
# ¿Coincide con la predicción del GIL (sin speedup, posible slowdown)?

def tarea_cpu_bound(n: int) -> int:
    """Tarea CPU-bound pura: wait(τᵢ) = ∅"""
    return sum(range(n))

N_CPU = 30_000_000
N_HILOS = 4

# --- Secuencial (M1) ---
t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0

# --- M2: await secuencial ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

# --- IMPLEMENTACIÓN M3: Threading ---
hilos = []
t0 = time.perf_counter()

for i in range(N_HILOS):
    # Creamos hilos que comparten la memoria del proceso
    h = threading.Thread(target=tarea_cpu_bound, args=(N_CPU,))
    hilos.append(h)
    h.start()

for h in hilos:
    h.join()  # Bloquea hasta que todos los hilos terminen

t_threading = time.perf_counter() - t0

# --- Reporte de Resultados ---
print(f'M1 secuencial: {t_secuencial:.2f}s')
print(f'M3 threading ({N_HILOS} hilos): {t_threading:.2f}s')
print(f'Speedup M3:    {t_secuencial/t_threading:.2f}x')

# ANÁLISIS DEL RESULTADO:
# ¿Coincide con la predicción del GIL?
# Sí. Justamente se trata de un slowdonw, paso de 1 a .97

## Sección 4: Race condition reproducible + fix con Lock

Las condiciones de carrera son consecuencia de la memoria compartida en concurrencia.
Reproduce el problema y aplica la solución con `threading.Lock`.

In [ ]:
import threading

# TAREA 4.1 — Reproduce la race condition
N_INCREMENTOS = 100_000
N_HILOS_RACE = 4

# Sin lock — resultado no determinista
contador_sin_lock = [0]

def incrementar_sin_lock():
    for _ in range(N_INCREMENTOS):
        contador_sin_lock[0] += 1   # NO atómico: LOAD, ADD, STORE separados

hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

esperado = N_INCREMENTOS * N_HILOS_RACE
print(f'Sin lock  — esperado: {esperado:,}, obtenido: {contador_sin_lock[0]:,}')
print(f'Diferencia: {esperado - contador_sin_lock[0]:,} incrementos perdidos')
print()

# TAREA 4.2 — Fix con Lock

def incrementar_con_lock(lock: threading.Lock):
    for _ in range(N_INCREMENTOS):
        with lock:  # Adquiere el lock antes de modificar el contador
            contador_con_lock[0] += 1   

lock = threading.Lock()
contador_con_lock = [0]

hilos = [threading.Thread(target=incrementar_con_lock, args=(lock,)) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

esperado = N_INCREMENTOS * N_HILOS_RACE
print(f'Con lock   — esperado: {esperado:,}, obtenido: {contador_con_lock[0]:,}')
print(f'Diferencia: {esperado - contador_con_lock[0]:,} incrementos perdidos')

## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

Implementa el servidor chatbot v2 usando asyncio y mide su comportamiento con N usuarios.

In [ ]:
import asyncio
import time
import random

# Operaciones I/O-bound del chatbot (simuladas)
async def consultar_bd(user_id: int) -> list:
    """wait(τᵢ): I/O a base de datos — ~50ms"""
    await asyncio.sleep(0.05)
    return [f'historial de usuario {user_id}']

async def llamar_llm(historial: list) -> str:
    """wait(τᵢ): I/O a API del LLM — 1–2s variable"""
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f'respuesta para: {historial[-1]}'

async def handle_request(user_id: int) -> dict:
    """Una petición completa del chatbot v2 (M4)"""
    # exec(τᵢ)
    t_inicio = time.perf_counter()

    # wait(τᵢ): BD
    historial = await consultar_bd(user_id)

    # wait(τᵢ): LLM
    respuesta = await llamar_llm(historial)

    # exec(τᵢ)
    latencia = time.perf_counter() - t_inicio
    return {'user': user_id, 'respuesta': respuesta, 'latencia': latencia}

# TAREA 5.1 — Servidor secuencial (chatbot v1 como baseline)
async def servidor_v1(n_usuarios: int):
    """M1: un usuario a la vez"""
    resultados = []
    for i in range(n_usuarios):
        r = await handle_request(i)
        resultados.append(r)
    return resultados

# TAREA 5.2 — Servidor concurrente (chatbot v2)
async def servidor_v2(n_usuarios: int):
    """M4: todos los usuarios concurrentes con gather"""
    resultados = await asyncio.gather(
        *[handle_request(i) for i in range(n_usuarios)]
    )
    return resultados



# TAREA 5.3 — Compara v1 vs v2 con N=10 usuarios
# Mide tiempos totales y latencias promedio
# ¿Cuántas veces más rápido es v2?
# ¿La latencia de cada usuario es similar en v2? ¿Por qué?

N = 10# --- Medición V1 (Secuencial - M1) ---

t0 = time.perf_counter()
res_v1 = await servidor_v1(N)
t_v1 = time.perf_counter() - t0
latencia_promedio_v1 = sum(r['latencia'] for r in res_v1) / N   

# --- Medición V2 (Concurrente - M4) ---
t0 = time.perf_counter()
res_v2 = await servidor_v2(N)
t_v2 = time.perf_counter() - t0
latencia_promedio_v2 = sum(r['latencia'] for r in res_v2) / N

# --- Reporte de resultados ---
print(f"=== RESULTADOS CON {N} USUARIOS ===")
print(f"V1 (Secuencial):  {t_v1:.2f}s total | Latencia promedio: {latencia_promedio_v1:.2f}s")
print(f"V2 (Concurrente): {t_v2:.2f}s total | Latencia promedio: {latencia_promedio_v2:.2f}s")
print(f"\nSpeedup total: {t_v1/t_v2:.1f}x más rápido")